#### **ОПИСАНИЕ ЗАДАНИЯ**
Найдите описание **CVE-уязвимости** (common vulnerabilities and exposures) и напишие минимальный **PoC-скрипт** (proof of concept), который эмулирует использование уязвимости (можно без её активации). В скрипте достаточно имитировать взаимодействие, например, формирование запроса к условной уязвимой точке, и выводить сообщение о потенциальной атаке.

##### **Выбранная уязвимость: CVE-2020-25213**
Недостаточная фильтрация входных данных в плагине *File Manager* для WordPress позволяла злоумышленникам читать произвольные файлы на сервере (например, wp-config.php, содержащий данные для доступа к базе данных), передавая специально сформированные параметры в URL. Эта уязвимость по типу относится к *Path Traversal* или *Обход пути*

Ссылка: [*кликабельно*](https://nvd.nist.gov/vuln/detail/CVE-2020-25213)

In [1]:
import requests
from urllib.parse import urljoin

In [2]:
def check_cve_2020_25213(base_url: str):
    """
    Проверяет наличие уязвимого файла connector.minimal.php.

    Согласно описанию CVE, переименование unsafe-файла в .php
    открывает возможность выполнения команд elFinder.

    Args:
        base_url (str): Адрес сайта (например, http://localhost:8000)
    """
    # Путь к уязвимому файлу, указанный в описании CVE
    vulnerable_path = (
        "wp-content/plugins/wp-file-manager/lib/php/connector.minimal.php"
    )
    
    target_url = urljoin(base_url, vulnerable_path)
    
    print(f"[*] Инициализация проверки цели: {base_url}")
    print(f"[*] Проверка доступности эндпоинта: {vulnerable_path}")

    try:
        # Отправляем GET-запрос. В реальной атаке здесь был бы POST с файлом.
        # Мы только проверяем наличие "дыры", чтобы не навредить.
        response = requests.get(target_url, timeout=10)

        # Если файл существует и сервер его обрабатывает (код 200),
        # значит, интерфейс elFinder открыт для всех.
        if response.status_code == 200:
            print("\n[!!!] УЯЗВИМОСТЬ ОБНАРУЖЕНА [!!!]")
            print(f"Адрес {target_url} доступен (HTTP 200).")
            print("Злоумышленник может использовать этот файл для RCE.")
            
            # Дополнительная проверка: часто elFinder отдает JSON-ответ "error" 
            # при пустом запросе, что подтверждает работу скрипта.
            if "json" in response.headers.get("Content-Type", ""):
                print("Тип контента подтверждает работу API elFinder.")
                
        elif response.status_code == 404:
            print("\n[-] Уязвимость не найдена.")
            print("Файл connector.minimal.php отсутствует или скрыт.")
            
        else:
            print(f"\n[-] Получен неожиданный код ответа: {response.status_code}")

    except requests.RequestException as error:
        print(f"\n[!] Ошибка соединения: {error}")

In [ ]:
# Пример использования
# Развернутый сервер на виртуальной машине Ubuntu 22.04
# Версия плагина: 6.0
TEST_TARGET = "http://192.168.1.151/wordpress/" 

check_cve_2020_25213(TEST_TARGET)

[*] Инициализация проверки цели: http://192.168.1.151/wordpress/
[*] Проверка доступности эндпоинта: wp-content/plugins/wp-file-manager/lib/php/connector.minimal.php

[!!!] УЯЗВИМОСТЬ ОБНАРУЖЕНА [!!!]
Адрес http://192.168.1.151/wordpress/wp-content/plugins/wp-file-manager/lib/php/connector.minimal.php доступен (HTTP 200).
Злоумышленник может использовать этот файл для RCE.
Тип контента подтверждает работу API elFinder.
